# 04 — Statistical Analysis & Figures

In this notebook, we perform the final statistical analysis and generate all 8 protocol figures.
The statistical unit is defined as a specific combination of `(dataset × base_size × split_seed)`.
Since we evaluate multiple hash seeds for approximate methods (SimHash, E2LSH), we aggregate over hash seeds first (taking the mean for quality metrics and the median for latency) before performing paired tests against the linear baseline.

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
import glob

sns.set_theme(style="whitegrid")
plt.rcParams.update({'figure.max_open_warning': 0})


In [ ]:
BENCHMARK_DIR = Path("../results/benchmark")
FIG_DIR = Path("../results/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

def load_csvs(pattern):
    files = list(BENCHMARK_DIR.glob(pattern))
    if not files:
        return pd.DataFrame()
    dfs = [pd.read_csv(f) for f in files]
    return pd.concat(dfs, ignore_index=True)

df_quality = load_csvs("benchmark_quality_seed*.csv")
df_timing = load_csvs("benchmark_timing_seed*.csv")
df_candidates = load_csvs("benchmark_candidates_seed*.csv")
df_memory = load_csvs("benchmark_memory_seed*.csv")

print(f"Quality rows: {len(df_quality)}")
print(f"Timing rows: {len(df_timing)}")
print(f"Candidates rows: {len(df_candidates)}")
print(f"Memory rows: {len(df_memory)}")


In [ ]:
# Join quality and timing data
if len(df_quality) > 0 and len(df_timing) > 0:
    df_merged = pd.merge(
        df_quality, df_timing, 
        on=["dataset", "base_size", "split_seed", "hash_seed", "method"], 
        how="outer"
    )
else:
    df_merged = pd.DataFrame(columns=["dataset", "base_size", "split_seed", "hash_seed", "method", "recall@10", "macro_f1", "median_latency_s"])

# Hash seed aggregation
# For linear, hash_seed is usually NaN or a placeholder, we group by it anyway or fill it
df_merged['hash_seed'] = df_merged['hash_seed'].fillna(-1)

df_agg = df_merged.groupby(["dataset", "base_size", "split_seed", "method"]).agg({
    "recall@1": "mean",
    "recall@5": "mean",
    "recall@10": "mean",
    "macro_f1": "mean",
    "median_latency_s": "median",
    "build_time_s": "mean"
}).reset_index()

print("Aggregated over hash seeds:")
display(df_agg.head())


### Statistical Tests
With only 5 splits, p-values from Wilcoxon tests are interpreted cautiously. We rely more on the absolute differences and the 95% Bootstrap Confidence Intervals.

In [ ]:
def bootstrap_ci(differences, n_bootstrap=1000, alpha=0.05, seed=42):
    rng = np.random.default_rng(seed)
    boot = [rng.choice(differences, size=len(differences), replace=True).mean()
            for _ in range(n_bootstrap)]
    lo, hi = np.percentile(boot, [100*alpha/2, 100*(1-alpha/2)])
    return float(np.mean(differences)), lo, hi


In [ ]:
sig_results = []
datasets = df_agg["dataset"].unique() if not df_agg.empty else []
base_sizes = df_agg["base_size"].unique() if not df_agg.empty else []
metrics = ["recall@10", "macro_f1", "median_latency_s"]

for ds in datasets:
    for bs in base_sizes:
        df_sub = df_agg[(df_agg["dataset"] == ds) & (df_agg["base_size"] == bs)]
        if df_sub.empty:
            continue
        
        linear_df = df_sub[df_sub["method"] == "linear"].sort_values("split_seed")
        for method in ["SimHash", "E2LSH"]:
            method_df = df_sub[df_sub["method"] == method].sort_values("split_seed")
            
            # Ensure we have paired data
            common_seeds = set(linear_df["split_seed"]).intersection(set(method_df["split_seed"]))
            if len(common_seeds) < 3:
                continue
                
            lin = linear_df[linear_df["split_seed"].isin(common_seeds)][metrics].values
            met = method_df[method_df["split_seed"].isin(common_seeds)][metrics].values
            
            for i, metric in enumerate(metrics):
                diff = met[:, i] - lin[:, i]
                try:
                    res = stats.wilcoxon(diff)
                    pval = res.pvalue
                except ValueError:
                    pval = 1.0
                
                mean_diff, ci_lo, ci_hi = bootstrap_ci(diff)
                sig_results.append({
                    "dataset": ds,
                    "base_size": bs,
                    "method": method,
                    "metric": metric,
                    "mean_diff": mean_diff,
                    "ci_lower": ci_lo,
                    "ci_upper": ci_hi,
                    "p_value": pval
                })

significance_tests_df = pd.DataFrame(sig_results)
if not significance_tests_df.empty:
    display(significance_tests_df.head())


In [ ]:
# Outcome classification
outcomes = []
if not significance_tests_df.empty:
    for ds in datasets:
        for bs in base_sizes:
            for method in ["SimHash", "E2LSH"]:
                df_sub = significance_tests_df[(significance_tests_df["dataset"] == ds) & 
                                               (significance_tests_df["base_size"] == bs) & 
                                               (significance_tests_df["method"] == method)]
                if df_sub.empty:
                    continue
                
                # Get absolute aggregated values to check recall@10 >= 0.95
                df_abs = df_agg[(df_agg["dataset"] == ds) & 
                                (df_agg["base_size"] == bs) & 
                                (df_agg["method"] == method)]
                linear_abs = df_agg[(df_agg["dataset"] == ds) & 
                                    (df_agg["base_size"] == bs) & 
                                    (df_agg["method"] == "linear")]
                
                if df_abs.empty or linear_abs.empty:
                    continue
                    
                recall = df_abs["recall@10"].mean()
                f1_drop = -df_sub[df_sub["metric"] == "macro_f1"]["mean_diff"].values[0]
                lat_diff = df_sub[df_sub["metric"] == "median_latency_s"]["mean_diff"].values[0]
                
                is_faster = lat_diff < 0
                quality_ok = (recall >= 0.95) and (f1_drop <= 0.01)
                
                if quality_ok and is_faster:
                    cls = "Gain confirmed" # Will refine to conditional later if needed
                elif quality_ok and not is_faster:
                    cls = "Quality preserved, no gain"
                elif not quality_ok and is_faster:
                    cls = "Quality failure"
                else:
                    cls = "Global failure"
                    
                outcomes.append({
                    "dataset": ds,
                    "base_size": bs,
                    "method": method,
                    "classification": cls
                })

df_outcomes = pd.DataFrame(outcomes)
if not df_outcomes.empty:
    display(df_outcomes)


In [ ]:
# Figure 1: Latency comparison
if not df_agg.empty:
    plt.figure(figsize=(10, 6))
    df_agg['ds_bs'] = df_agg['dataset'] + '_' + df_agg['base_size'].astype(str)
    sns.barplot(data=df_agg, x='ds_bs', y='median_latency_s', hue='method', errorbar=('ci', 95))
    plt.title("Figure 1: Median Latency Comparison")
    plt.ylabel("Median Latency (s)")
    plt.xlabel("Dataset & Base Size")
    plt.yscale('log')
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig01_latency.png", dpi=300)
    plt.show()


In [ ]:
# Figure 2: Recall@k
if not df_agg.empty:
    df_melt = df_agg.melt(id_vars=["dataset", "base_size", "split_seed", "method"], 
                          value_vars=["recall@1", "recall@5", "recall@10"], 
                          var_name="k", value_name="recall")
    g = sns.catplot(data=df_melt, x="k", y="recall", hue="method", col="dataset", 
                    kind="bar", errorbar=('ci', 95), height=5, aspect=1.2)
    g.fig.suptitle("Figure 2: Recall@k Across Datasets", y=1.05)
    plt.savefig(FIG_DIR / "fig02_recall.png", dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
# Figure 3: Candidate fraction
if len(df_candidates) > 0:
    plt.figure(figsize=(10, 6))
    df_candidates['ds_bs'] = df_candidates['dataset'] + '_' + df_candidates['base_size'].astype(str)
    sns.barplot(data=df_candidates, x='ds_bs', y='candidate_fraction', hue='method')
    plt.title("Figure 3: Candidate Fraction")
    plt.ylabel("Fraction of Base Searched")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig03_candidates.png", dpi=300)
    plt.show()


In [ ]:
# Figure 4: Build time
if not df_agg.empty:
    plt.figure(figsize=(10, 6))
    sns.barplot(data=df_agg, x='ds_bs', y='build_time_s', hue='method', errorbar=('ci', 95))
    plt.title("Figure 4: Index Build Time")
    plt.ylabel("Build Time (s)")
    plt.yscale('log')
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig04_buildtime.png", dpi=300)
    plt.show()


In [ ]:
# Figure 5: Memory
if len(df_memory) > 0:
    plt.figure(figsize=(10, 6))
    df_memory['ds_bs'] = df_memory['dataset'] + '_' + df_memory['base_size'].astype(str)
    sns.barplot(data=df_memory, x='ds_bs', y='index_memory_bytes', hue='method')
    plt.title("Figure 5: Memory Usage")
    plt.ylabel("Index Memory (bytes)")
    plt.yscale('log')
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig05_memory.png", dpi=300)
    plt.show()


In [ ]:
# Figure 6: Recall-Latency Trade-off
if not df_agg.empty:
    plt.figure(figsize=(8, 6))
    sns.scatterplot(data=df_agg, x='median_latency_s', y='recall@10', hue='method', style='dataset', s=100)
    plt.title("Figure 6: Recall-Latency Trade-off")
    plt.xscale('log')
    plt.xlabel("Median Latency (s)")
    plt.ylabel("Recall@10")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig06_recall_latency_tradeoff.png", dpi=300)
    plt.show()


In [ ]:
# Figure 7: Scaling (ChestX)
if not df_agg.empty and "ChestX" in datasets:
    df_chest = df_agg[df_agg['dataset'] == 'ChestX']
    if not df_chest.empty:
        plt.figure(figsize=(8, 6))
        sns.lineplot(data=df_chest, x='base_size', y='median_latency_s', hue='method', marker='o')
        plt.title("Figure 7: Scaling on ChestX")
        plt.xlabel("Base Size")
        plt.ylabel("Median Latency (s)")
        plt.xscale('log')
        plt.yscale('log')
        plt.tight_layout()
        plt.savefig(FIG_DIR / "fig07_scaling.png", dpi=300)
        plt.show()


In [ ]:
# Figure 8: Hash seed variance (E2LSH)
if not df_merged.empty:
    df_e2 = df_merged[df_merged['method'] == 'E2LSH'].copy()
    if not df_e2.empty:
        df_e2['ds_bs'] = df_e2['dataset'] + '_' + df_e2['base_size'].astype(str)
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=df_e2, x='ds_bs', y='recall@10')
        sns.stripplot(data=df_e2, x='ds_bs', y='recall@10', color="orange", size=6, alpha=0.7)
        plt.title("Figure 8: Hash Seed Variance for E2LSH")
        plt.ylabel("Recall@10")
        plt.tight_layout()
        plt.savefig(FIG_DIR / "fig08_hash_seed_variance.png", dpi=300)
        plt.show()


In [ ]:
# Save outputs
if not significance_tests_df.empty:
    significance_tests_df.to_csv(BENCHMARK_DIR / "significance_tests.csv", index=False)

summary_data = {
    "outcome_by_method": {},
    "overall_conclusion": "Detailed conclusion based on tests.",
    "best_config_per_dataset": {}
}
if 'df_outcomes' in locals() and not df_outcomes.empty:
    for method in ["SimHash", "E2LSH"]:
        method_df = df_outcomes[df_outcomes["method"] == method]
        classes = method_df["classification"].value_counts().to_dict()
        summary_data["outcome_by_method"][method] = classes

with open(BENCHMARK_DIR / "summary.json", "w") as f:
    json.dump(summary_data, f, indent=4)


### Conclusion

Based on the protocol outcomes classification table, we can determine whether LSH provides a consistent speedup with >95% recall, conditionally works well, or fails either in quality or latency when applied to the embedding similarities in these specific datasets.